# 01 — Task 1: FER2013 baseline (20 %)

* **Classifiers:** Logistic Regression, SVM (RBF kernel), Random Forest.
* **Features:** HOG, dlib facial landmarks, PCA of the raw pixels (eigenfaces), and HOG+landmarks.
* **Protocol:** Training split for fitting, PublicTest (validation) for model selection and
  hyper-parameter tuning, PrivateTest (test) **only** for the final reported numbers.
  Every grid search uses a `PredefinedSplit` (train → validation) with macro-F1 as score, because
  FER is imbalanced and accuracy alone would reward ignoring *Disgust*.
* **Metrics:** accuracy, balanced accuracy, macro-F1, confusion matrix.
* **Preprocessing study (gray vs black-and-white):** the best classifier is re-trained on five
  image variants — gray, CLAHE, Otsu binarization, adaptive binarization, edge map.
* **Error analysis:** hardest emotions, most confused pairs, lighting/contrast, confident errors
  (likely mislabels), duplicates between train and test.

All result tables have the same layout in every notebook: rows = classifier (LR, SVM, RF),
columns = feature (HOG, LM, PCA, HOG+LM).

In [ ]:
# Common setup + evaluation helpers (identical in every notebook of the project)
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedShuffleSplit

sys.path.insert(0, str(Path.cwd()))
try:
    from src import features as F
except ImportError:  # fuzzy environment (no OpenCV/dlib): only cached features are used there
    from types import SimpleNamespace
    F = SimpleNamespace(EMOTIONS=["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"],
                        CACHE_DIR=Path("data") / "cache")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# SMOKE=1 (environment variable) runs everything on small subsets with reduced grids, to
# check the notebook end-to-end in minutes. The delivered results use SMOKE=0 (default).
SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
N_JOBS = -1
EMO = F.EMOTIONS
RESULTS_DIR, FIG_DIR, MODEL_DIR = Path("results"), Path("figures"), Path("models")
for d in (RESULTS_DIR, FIG_DIR, MODEL_DIR, F.CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "savefig.bbox": "tight"})
print(f"SMOKE={SMOKE}")


def metrics(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred),
            "bal_acc": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", labels=range(7), zero_division=0)}


def per_class_f1(y_true, y_pred):
    return pd.Series(f1_score(y_true, y_pred, average=None, labels=range(7), zero_division=0), index=EMO)


_RESULTS = {}


def reset_results(task):
    """Start a fresh results/<task>.csv for this run of the notebook."""
    _RESULTS[task] = []
    (RESULTS_DIR / f"{task}.csv").unlink(missing_ok=True)


def log_result(task, y_true, y_pred, **info):
    """Append one evaluation row (fixed schema) to results/<task>.csv and return it."""
    row = {**info, **{k: round(v, 4) for k, v in metrics(y_true, y_pred).items()}, "n": len(y_true)}
    _RESULTS.setdefault(task, []).append(row)
    pd.DataFrame(_RESULTS[task]).to_csv(RESULTS_DIR / f"{task}.csv", index=False)
    return row


def fixed_table(df, index, columns, value, index_order, column_order):
    """Pivot to a table with a FIXED row/column layout (same x and y in every task)."""
    t = df.pivot_table(index=index, columns=columns, values=value, aggfunc="first")
    return t.reindex(index=index_order, columns=column_order)


def save_table(df, name, float_format="%.3f"):
    """Save a table as CSV and as a LaTeX tabular (pasted into report/*.md)."""
    df.to_csv(RESULTS_DIR / f"{name}.csv")
    with open(RESULTS_DIR / f"{name}.tex", "w", encoding="utf-8") as f:
        f.write(df.to_latex(float_format=float_format, na_rep="--"))
    return df


def plot_cm(y_true, y_pred, title="", ax=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=range(7), normalize="true" if normalize else None)
    ax = ax or plt.subplots(figsize=(4.2, 3.6))[1]
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues", cbar=False,
                xticklabels=[e[:3] for e in EMO], yticklabels=[e[:3] for e in EMO], ax=ax,
                vmin=0, vmax=1 if normalize else None, annot_kws={"size": 7})
    ax.set_title(title, fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    return ax


def plot_cm_grid(items, fname, ncols=4):
    """items: list of (title, y_true, y_pred). Saved in figures/ for the report appendix."""
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.7 * nrows), squeeze=False)
    for ax, (title, yt, yp) in zip(axes.ravel(), items):
        plot_cm(yt, yp, title, ax=ax)
    for ax in axes.ravel()[len(items):]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()


def stratified_subset(y, n, seed=SEED):
    """Indices of a stratified random subset of size n (all indices if n >= len(y))."""
    if n is None or n >= len(y):
        return np.arange(len(y))
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    return np.sort(next(sss.split(np.zeros(len(y)), y))[0])


def scores(model, X):
    """Class scores for confidence analyses (predict_proba, else softmax(decision_function))."""
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)
        except AttributeError:
            pass
    d = model.decision_function(X)
    e = np.exp(d - d.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def save_bundle(name, model, spec, **extra):
    """Persist a FITTED model together with the feature spec needed to use it (demo, other tasks)."""
    bundle = {"model": model, "spec": spec, "labels": EMO, **extra}
    joblib.dump(bundle, MODEL_DIR / f"{name}.pkl", compress=3)
    print(f"saved models/{name}.pkl")


def show_images(imgs, titles=None, ncols=8, size=1.5, fname=None, suptitle=None, cmap="gray"):
    nrows = int(np.ceil(len(imgs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * 1.15 * nrows), squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < len(imgs):
            ax.imshow(imgs[i], cmap=None if imgs[i].ndim == 3 else cmap, vmin=0, vmax=255)
            if titles is not None:
                ax.set_title(titles[i], fontsize=7)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    if fname:
        fig.savefig(FIG_DIR / fname)
    plt.show()

In [ ]:
# Classifiers, feature sets and the tuning protocol (shared by notebooks 01, 02, 03, 05)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit, GroupKFold
from sklearn.base import clone

MODELS = ["LR", "SVM", "RF"]               # fixed row order of every result grid
FEATURES = ["HOG", "LM", "PCA", "HOG+LM"]   # fixed column order of every result grid
FEATURE_BLOCKS = {"HOG": ["hog"], "LM": ["lm"], "PCA": ["pix"], "HOG+LM": ["hog", "lm"]}
N_TREES = 100 if SMOKE else 300


def base_estimator(model, n_jobs=1):
    if model == "LR":
        return LogisticRegression(max_iter=3000, random_state=SEED)
    if model == "SVM":
        return SVC(kernel="rbf", gamma="scale", cache_size=2000, random_state=SEED)
    if model == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", n_jobs=n_jobs, random_state=SEED)
    raise ValueError(model)


def make_pipeline(model, feature, n_jobs=1):
    """StandardScaler -> [PCA (eigenfaces, only for raw pixels)] -> classifier."""
    steps = [("scaler", StandardScaler())]
    if feature == "PCA":
        steps.append(("pca", PCA(n_components=100, whiten=True, random_state=SEED)))
    steps.append(("clf", base_estimator(model, n_jobs)))
    return Pipeline(steps)


if SMOKE:
    GRIDS = {"LR": {"clf__C": [0.1, 1.0], "clf__class_weight": ["balanced"]},
             "SVM": {"clf__C": [1, 10], "clf__class_weight": ["balanced"]},
             "RF": {"clf__max_depth": [None], "clf__class_weight": [None, "balanced_subsample"]}}
    PCA_GRID = {"pca__n_components": [60]}
else:
    GRIDS = {"LR": {"clf__C": [0.01, 0.1, 1.0], "clf__class_weight": [None, "balanced"]},
             "SVM": {"clf__C": [1, 3, 10], "clf__class_weight": [None, "balanced"]},
             "RF": {"clf__max_depth": [None, 25], "clf__min_samples_leaf": [1, 3],
                    "clf__class_weight": [None, "balanced_subsample"]}}
    PCA_GRID = {"pca__n_components": [60, 120]}


def param_grid(model, feature):
    return {**GRIDS[model], **(PCA_GRID if feature == "PCA" else {})}


def tune_predefined(model, feature, X_tr, y_tr, X_va, y_va, n_search):
    """Grid search: fit on (a stratified subset of) Training, score macro-F1 on PublicTest."""
    idx = stratified_subset(y_tr, n_search)
    X = np.vstack([X_tr[idx], X_va]); y = np.concatenate([y_tr[idx], y_va])
    ps = PredefinedSplit(np.concatenate([-np.ones(len(idx)), np.zeros(len(y_va))]))
    gs = GridSearchCV(make_pipeline(model, feature), param_grid(model, feature),
                      scoring="f1_macro", cv=ps, n_jobs=N_JOBS, refit=False)
    gs.fit(X, y)
    return gs.best_params_, gs.best_score_


def tune_groupkfold(model, feature, X, y, groups, n_splits=5):
    """Subject-independent CV (no validation split exists for KDEF)."""
    gs = GridSearchCV(make_pipeline(model, feature), param_grid(model, feature), scoring="f1_macro",
                      cv=GroupKFold(n_splits=n_splits), n_jobs=N_JOBS, refit=False)
    gs.fit(X, y, groups=groups)
    std = gs.cv_results_["std_test_score"][gs.best_index_]
    return gs.best_params_, gs.best_score_, std


def fit_final(model, feature, params, X, y, sample_weight=None):
    pipe = make_pipeline(model, feature, n_jobs=N_JOBS).set_params(**params)
    fit_kw = {"clf__sample_weight": sample_weight} if sample_weight is not None else {}
    return pipe.fit(X, y, **fit_kw)

In [ ]:
# Cached datasets and features (written by 00_data_preparation.ipynb)
# Dataset keys: "fer_gray", "fer_aligned", "kdef_m0.0_gray" ... "kdef_m0.3_gray", "kdef_m0.2_rgb",
# "kdef_m0.2_gray_aligned". Splits: FER train/val/test, KDEF train/test.
FEATURE_BLOCKS.update({"GEO": ["geo"]})
_DS = {}


def images(key, split):
    if key not in _DS:
        _DS[key] = dict(np.load(F.CACHE_DIR / f"{key}.npz", allow_pickle=True))
    return _DS[key][f"X_{split}"], _DS[key][f"y_{split}"]


def meta(key, field, split):
    images(key, split)
    return _DS[key][f"{field}_{split}"]


def landmarks(key, split, variant="gray"):
    X, _ = images(key, split)
    vt = "" if variant == "gray" else f"-{variant}"
    return F.cached(f"{key}{vt}_{split}_pts", lambda: F.extract_landmarks(F.apply_variant(X, variant)))


def features(key, split, feature, variant="gray", hog=None):
    """Feature matrix of a cached dataset split (computed once, then loaded from data/cache)."""
    if feature == "HOG+LM":
        return np.hstack([features(key, split, "HOG", variant, hog), features(key, split, "LM", variant)])
    X, _ = images(key, split)
    vt = "" if variant == "gray" else f"-{variant}"
    ht = "" if not hog else "_o{}c{}".format(hog["orientations"], hog["pixels_per_cell"][0])
    spec = {"variant": variant, "blocks": FEATURE_BLOCKS[feature], "hog": hog or {}}
    pts = landmarks(key, split, variant) if feature in ("LM", "GEO") else None
    return F.cached(f"{key}{vt}_{split}_{feature}{ht}", lambda: F.extract_feature_set(X, spec, pts=pts))


def spec_for(feature, variant="gray", hog=None):
    """Feature spec stored in a model bundle (lets demo_emotion.py rebuild the features)."""
    blocks = sum((FEATURE_BLOCKS[f] for f in feature.split("+")), [])
    return {"variant": variant, "blocks": blocks, "hog": hog or {}, "feature": feature}

In [ ]:
reset_results("task1_grid"); reset_results("task1_preproc"); reset_results("task1_baseline")
y_tr_full, y_va_full, y_te = images("fer_gray", "train")[1], images("fer_gray", "val")[1], images("fer_gray", "test")[1]

# SMOKE uses small stratified subsets; the full run uses everything.
TR = stratified_subset(y_tr_full, 3000 if SMOKE else None)
VA = stratified_subset(y_va_full, 1000 if SMOKE else None)
SEARCH_N = 2000 if SMOKE else 10000   # grid search on a stratified 10k subset of train (SVM cost)
STUDY_N = 2000 if SMOKE else 10000    # train size of the preprocessing study
y_tr, y_va = y_tr_full[TR], y_va_full[VA]
print(f"train {len(y_tr)}  val {len(y_va)}  test {len(y_te)}")


def fer_split(feature, variant="gray", key="fer_gray", hog=None):
    """(X_train, X_val, X_test) for one feature set, restricted to the SMOKE subsets."""
    return (features(key, "train", feature, variant, hog)[TR],
            features(key, "val", feature, variant, hog)[VA],
            features(key, "test", feature, variant, hog))


t = time.time()
FX = {f: fer_split(f) for f in FEATURES}
print({f: X[0].shape[1] for f, X in FX.items()}, f"features ready in {time.time() - t:.0f}s")

## 1. Starting point: the kit's baseline (Random Forest + landmarks)
Same hyper-parameters as `baseline_fer2013_landmarks.ipynb` (300 trees, depth 11, entropy).
The kit reports accuracy 0.44 / balanced accuracy 0.37 / macro-F1 0.38 on the validation set.

In [ ]:
kit_rf = Pipeline([("clf", RandomForestClassifier(n_estimators=300, max_depth=11, max_features="sqrt",
                                                  criterion="entropy", random_state=SEED, n_jobs=N_JOBS))])
kit_rf.fit(FX["LM"][0], y_tr)
for split, X, y in [("val", FX["LM"][1], y_va), ("test", FX["LM"][2], y_te)]:
    print(split, log_result("task1_baseline", y, kit_rf.predict(X), model="RF (kit)", feature="LM", split=split))

## 2. Classifier × feature grid
For each of the 12 combinations: grid search on train→validation, refit on the full training
set with the best parameters, evaluate on validation and test.
* LR: `C`, class weighting. * SVM (RBF): `C`, class weighting (`gamma='scale'`).
* RF: depth, min. leaf size, class weighting. * PCA: number of components (whitened).

In [ ]:
fitted, preds, best_params = {}, {}, {}
for model in MODELS:
    for feat in FEATURES:
        t = time.time()
        X_tr, X_va, X_te = FX[feat]
        params, cv_score = tune_predefined(model, feat, X_tr, y_tr, X_va, y_va, SEARCH_N)
        pipe = fit_final(model, feat, params, X_tr, y_tr)
        fitted[(model, feat)], best_params[(model, feat)] = pipe, params
        for split, X, y in [("val", X_va, y_va), ("test", X_te, y_te)]:
            p = pipe.predict(X)
            preds[(model, feat, split)] = p
            log_result("task1_grid", y, p, model=model, feature=feat, split=split,
                       search_score=round(cv_score, 4), params=json.dumps(params))
        print(f"{model:4s} {feat:7s} {params}  val-search F1={cv_score:.3f}  "
              f"test F1={f1_score(y_te, preds[(model, feat, 'test')], average='macro'):.3f}  ({time.time() - t:.0f}s)")

In [ ]:
res = pd.read_csv(RESULTS_DIR / "task1_grid.csv")
tables = {}
for split in ("val", "test"):
    for m in ("macro_f1", "acc", "bal_acc"):
        t = fixed_table(res[res.split == split], "model", "feature", m, MODELS, FEATURES)
        tables[(split, m)] = save_table(t, f"task1_grid_{split}_{m}")
for m in ("macro_f1", "acc", "bal_acc"):
    print(f"\nPrivateTest {m}"); display(tables[("test", m)].round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, m in zip(axes, ("macro_f1", "acc", "bal_acc")):
    sns.heatmap(tables[("test", m)], annot=True, fmt=".3f", cmap="viridis", ax=ax, cbar=False)
    ax.set_title(f"FER PrivateTest {m}")
fig.tight_layout(); fig.savefig(FIG_DIR / "task1_grid_heatmaps.png"); plt.show()

plot_cm_grid([(f"{m} / {f}", y_te, preds[(m, f, "test")]) for m in MODELS for f in FEATURES],
             "appendix_task1_confusion_matrices.png", ncols=4)

### Model selection (validation set only)
The best feature set per classifier and the overall best configuration are selected on the
**validation** macro-F1; the test numbers above are only reported.

In [ ]:
val_f1 = tables[("val", "macro_f1")]
best_feat_per_model = val_f1.idxmax(axis=1).to_dict()
BEST_MODEL, BEST_FEAT = val_f1.stack().idxmax()
best_feature_overall = val_f1.mean(axis=0).idxmax()
print("best feature per classifier (val):", best_feat_per_model)
print("best configuration (val):", BEST_MODEL, BEST_FEAT, best_params[(BEST_MODEL, BEST_FEAT)])
print("best feature on average over classifiers (val):", best_feature_overall)

for model, feat in best_feat_per_model.items():
    save_bundle(f"task1_{model}", fitted[(model, feat)], spec_for(feat), params=best_params[(model, feat)],
                trained_on="FER2013 train", task="task1")
json.dump({"model": BEST_MODEL, "feature": BEST_FEAT, "params": best_params[(BEST_MODEL, BEST_FEAT)],
           "best_feature_per_model": best_feat_per_model, "best_feature_overall": best_feature_overall,
           "all_params": {f"{m}|{f}": p for (m, f), p in best_params.items()}},
          open(RESULTS_DIR / "task1_best.json", "w"), indent=2)

## 3. Preprocessing study: gray levels vs black-and-white vs edges
Question: is the expression carried by the grey-level shading, or is the *shape* (the binary
"kernel" of the face) enough? The best classifier keeps its tuned hyper-parameters for each
feature and is re-trained on a stratified subset of `STUDY_N` training images for every variant.

In [ ]:
VARIANT_ORDER = ["gray", "clahe", "otsu", "adaptive", "edges"]
STUDY_FEATS = ["HOG", "LM", "PCA"]
idx_study = stratified_subset(y_tr, STUDY_N)
for variant in VARIANT_ORDER:
    for feat in STUDY_FEATS:
        X_tr, X_va, X_te = fer_split(feat, variant)
        pipe = fit_final(BEST_MODEL, feat, best_params[(BEST_MODEL, feat)], X_tr[idx_study], y_tr[idx_study])
        for split, X, y in [("val", X_va, y_va), ("test", X_te, y_te)]:
            log_result("task1_preproc", y, pipe.predict(X), model=BEST_MODEL, feature=feat,
                       variant=variant, split=split)
    print("done", variant)

pre = pd.read_csv(RESULTS_DIR / "task1_preproc.csv")
for m in ("macro_f1", "acc"):
    t = save_table(fixed_table(pre[pre.split == "test"], "variant", "feature", m, VARIANT_ORDER, STUDY_FEATS),
                   f"task1_preproc_test_{m}")
    print(f"PrivateTest {m} ({BEST_MODEL}, {len(idx_study)} training images)"); display(t.round(3))

## 4. Error analysis of the best configuration
### 4.1 Per-class performance and confusions

In [ ]:
best = fitted[(BEST_MODEL, BEST_FEAT)]
X_te_best = FX[BEST_FEAT][2]
p_te = preds[(BEST_MODEL, BEST_FEAT, "test")]
print(classification_report(y_te, p_te, target_names=EMO, digits=3))
pcf = pd.DataFrame({f"{m}/{best_feat_per_model[m]}": per_class_f1(y_te, preds[(m, best_feat_per_model[m], "test")])
                    for m in MODELS})
save_table(pcf, "task1_per_class_f1")
pcf.plot.bar(figsize=(9, 3.2), title="FER PrivateTest per-class F1 (best feature per classifier)")
plt.ylabel("F1"); plt.tight_layout(); plt.savefig(FIG_DIR / "task1_per_class_f1.png"); plt.show()

cm = confusion_matrix(y_te, p_te, labels=range(7))
off = [(cm[i, j], EMO[i], EMO[j]) for i in range(7) for j in range(7) if i != j]
top_conf = pd.DataFrame(sorted(off, reverse=True)[:8], columns=["count", "true", "predicted"])
save_table(top_conf.set_index(["true", "predicted"]), "task1_top_confusions", float_format="%d")
display(top_conf)
plot_cm(y_te, p_te, f"FER test — {BEST_MODEL} / {BEST_FEAT}"); plt.savefig(FIG_DIR / "task1_best_cm.png"); plt.show()

### 4.2 Examples of the hardest emotions (misclassified test images)

In [ ]:
X_te_img = images("fer_gray", "test")[0]
hardest = per_class_f1(y_te, p_te).sort_values().index[:3]
for emo in hardest:
    c = EMO.index(emo)
    wrong = np.where((y_te == c) & (p_te != c))[0][:10]
    show_images(X_te_img[wrong], [f"→ {EMO[p_te[i]]}" for i in wrong], ncols=10, size=1.3,
                suptitle=f"True {emo}, misclassified", fname=f"task1_hard_{emo}.png")

### 4.3 Lighting and contrast
Test images are binned by mean intensity (exposure) and by pixel standard deviation (contrast).

In [ ]:
bright = X_te_img.reshape(len(X_te_img), -1).mean(axis=1)
contrast = X_te_img.reshape(len(X_te_img), -1).std(1)
rows = []
for name, v in [("brightness", bright), ("contrast", contrast)]:
    bins = pd.qcut(v, 5, labels=["very low", "low", "mid", "high", "very high"])
    for b in bins.categories:
        m = np.asarray(bins == b)
        rows.append({"property": name, "quintile": b, **metrics(y_te[m], p_te[m])})
light = pd.DataFrame(rows)
save_table(light.set_index(["property", "quintile"]), "task1_lighting")
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, name in zip(axes, ("brightness", "contrast")):
    light[light.property == name].plot.bar(x="quintile", y=["acc", "macro_f1"], ax=ax, title=f"test performance vs {name}", rot=0)
    ax.set_ylim(0, 1)
fig.tight_layout(); fig.savefig(FIG_DIR / "task1_lighting.png"); plt.show()

order = np.argsort(bright)
sel = np.r_[order[:6], order[-6:]]
show_images(X_te_img[sel], [f"{EMO[y_te[i]][:4]}→{EMO[p_te[i]][:4]}" for i in sel], ncols=6, size=1.4,
            suptitle="Darkest (top) and brightest (bottom) test images: true→predicted", fname="task1_lighting_examples.png")

### 4.4 Confident errors — candidate mislabelled images
Test images that the model misclassifies with the highest confidence. Many of them are visibly
mislabelled, ambiguous, non-faces or cartoons — FER's label noise.

In [ ]:
S = scores(best, X_te_best)
conf = S.max(1)
wrong = np.where(p_te != y_te)[0]
top = wrong[np.argsort(-conf[wrong])][:16]
show_images(X_te_img[top], [f"label {EMO[y_te[i]][:4]} / pred {EMO[p_te[i]][:4]}" for i in top], ncols=8, size=1.6,
            suptitle="Most confident errors on PrivateTest", fname="task1_confident_errors.png")

### 4.5 Train/test duplicates
Notebook 00 found test images that are pixel-identical to training images. Their accuracy is
compared with the rest of the test set — duplicates inflate FER test scores slightly.

In [ ]:
h_tr, h_te = F.dhash(images("fer_gray", "train")[0]), F.dhash(X_te_img)
is_dup = np.isin(h_te, h_tr)
dup_tab = pd.DataFrame([{"subset": "test images also in train", "n": int(is_dup.sum()), **metrics(y_te[is_dup], p_te[is_dup])},
                        {"subset": "other test images", "n": int((~is_dup).sum()), **metrics(y_te[~is_dup], p_te[~is_dup])}]).set_index("subset")
save_table(dup_tab, "task1_duplicates"); display(dup_tab.round(3))

## 5. Summary (auto-generated from this run)

In [ ]:
tf = tables[("test", "macro_f1")]
print(f"Best configuration (selected on validation): {BEST_MODEL} + {BEST_FEAT}")
print(f"  test acc={tables[('test', 'acc')].loc[BEST_MODEL, BEST_FEAT]:.3f}  "
      f"bal_acc={tables[('test', 'bal_acc')].loc[BEST_MODEL, BEST_FEAT]:.3f}  macro-F1={tf.loc[BEST_MODEL, BEST_FEAT]:.3f}")
print("Mean test macro-F1 per classifier:", tf.mean(axis=1).round(3).to_dict())
print("Mean test macro-F1 per feature:   ", tf.mean(axis=0).round(3).to_dict())
print("Hardest emotions (F1):", per_class_f1(y_te, p_te).sort_values().round(3).head(3).to_dict())
print("Preprocessing variants (test macro-F1, mean over features):",
      pre[pre.split == "test"].groupby("variant").macro_f1.mean().reindex(VARIANT_ORDER).round(3).to_dict())

**Interpretation guide (filled in the report with the numbers above).**
* *Classifier:* the RBF-SVM can model non-linear boundaries in the high-dimensional HOG space;
  LR is limited to linear boundaries; RF struggles with many weak, correlated HOG features.
* *Features:* HOG encodes local edge orientations (wrinkles, mouth/eye shape) and is robust to
  global illumination changes; landmarks are compact but suffer from dlib errors on 48×48 in-the-wild
  faces; PCA on raw pixels mixes identity, pose and illumination with expression.
* *Hardest emotions:* Disgust (few samples), Fear (confused with Surprise/Sad), Sad (confused
  with Neutral) — see the confusion matrix and galleries.
* *Black-and-white:* binarization discards shading information (nasolabial folds, brow
  furrows) and is extremely sensitive to lighting, so it is expected to lose performance;
  CLAHE / edges keep the shape and may help.